# DINOv2 Aerial Embeddings

Extracting DINOv2 embeddings from Digimap aerial image crops.

Steps:

1. Mount Google Drive and define paths.
2. Build/load the Digimap raster index from all JPG tiles.
3. Load the London-wide DINO sample and crop index.
4. Build a crop source index: `single_tile` or `mosaic`.
5. Load public DINOv2: `facebook/dinov2-base`.
6. Test PTAL/EPC × single/mosaic crops.
7. Extract embeddings in resumable chunks.
8. Combine chunks into one final Parquet file.


## 1. Setup


In [ ]:
!pip -q install rasterio pyarrow transformers accelerate tqdm


In [ ]:
import os
import re
import gc
import json
import math
import warnings
from pathlib import Path
from contextlib import ExitStack

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

import rasterio
from rasterio.windows import from_bounds
from rasterio.merge import merge

import torch
from transformers import AutoImageProcessor, AutoModel

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)


In [ ]:
# GPU check
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not available. Please switch Colab runtime to T4 GPU before continuing.")


## 2. Mount Google Drive and define project paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/GEOG0105")

DIGIMAP_DIR = BASE_DIR / "Raw Data" / "Digimap"

OUT_DIR = BASE_DIR / "Outputs"
TABLE_DIR = OUT_DIR / "tables"
EMB_DIR = OUT_DIR / "embeddings"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

RASTER_INDEX_PATH = TABLE_DIR / "digimap_raster_index_all.csv"
SOURCE_INDEX_PATH = TABLE_DIR / "aerial_crop_source_index_all.csv"

CHUNK_DIR = EMB_DIR / "dinov2_aerial_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

FINAL_EMB_PATH = EMB_DIR / "dinov2_aerial_embeddings_dino_sample.parquet"

print("BASE_DIR:", BASE_DIR)
print("DIGIMAP_DIR exists:", DIGIMAP_DIR.exists())
print("TABLE_DIR:", TABLE_DIR)
print("EMB_DIR:", EMB_DIR)


## 3. Auto-locate input CSV files

The crop index is the main input. The sample file is used only to recover missing metadata if needed.


In [ ]:
def find_file(base_dir, filename):
    matches = list(base_dir.rglob(filename))
    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {filename} under {base_dir}")
    matches = sorted(matches, key=lambda p: len(str(p)))
    return matches[0]

SAMPLE_FILE = find_file(BASE_DIR, "dino_londonwide_sample.csv")
CROP_INDEX_FILE = find_file(BASE_DIR, "aerial_crop_index_dino_londonwide_sample.csv")

print("SAMPLE_FILE:", SAMPLE_FILE)
print("CROP_INDEX_FILE:", CROP_INDEX_FILE)


In [ ]:
sample_df = pd.read_csv(SAMPLE_FILE)
crop_df = pd.read_csv(CROP_INDEX_FILE)

print("sample_df:", sample_df.shape)
print("crop_df:", crop_df.shape)

display(sample_df.head())
display(crop_df.head())


## 4. Build or load Digimap raster index

This scans all Digimap JPG tiles and records their bounds. It should find around 1,935 JPG files.


In [ ]:
def build_raster_index(digimap_dir, out_path):
    jpg_files = sorted(list(digimap_dir.rglob("*.jpg")) + list(digimap_dir.rglob("*.JPG")))
    print("JPG files found:", len(jpg_files))

    if len(jpg_files) == 0:
        raise FileNotFoundError(f"No JPG files found under {digimap_dir}")

    rows = []
    errors = []

    for p in tqdm(jpg_files, desc="Reading raster bounds"):
        try:
            with rasterio.open(p) as src:
                b = src.bounds
                res_x = abs(src.transform.a)
                res_y = abs(src.transform.e)
                rows.append({
                    "path": str(p),
                    "file": p.name,
                    "folder": p.parent.name,
                    "width": src.width,
                    "height": src.height,
                    "count": src.count,
                    "minx": float(b.left),
                    "miny": float(b.bottom),
                    "maxx": float(b.right),
                    "maxy": float(b.top),
                    "res_x": float(res_x),
                    "res_y": float(res_y),
                })
        except Exception as e:
            errors.append({"path": str(p), "error": repr(e)})

    raster_idx = pd.DataFrame(rows)
    raster_idx.to_csv(out_path, index=False)

    if errors:
        err_path = out_path.with_name(out_path.stem + "_errors.csv")
        pd.DataFrame(errors).to_csv(err_path, index=False)
        print("Raster read errors:", len(errors), "saved to", err_path)
    else:
        print("Raster read errors: 0")

    return raster_idx


def standardise_raster_index_columns(raster_idx):
    """
    Make old and new raster index formats consistent.
    Old format: left, bottom, right, top, pixel_width, pixel_height, name, bands
    New format: minx, miny, maxx, maxy, res_x, res_y, file, count
    """
    rename_map = {
        "left": "minx",
        "bottom": "miny",
        "right": "maxx",
        "top": "maxy",
        "pixel_width": "res_x",
        "pixel_height": "res_y",
        "name": "file",
        "bands": "count",
        "batch": "folder",
    }

    raster_idx = raster_idx.rename(
        columns={k: v for k, v in rename_map.items() if k in raster_idx.columns}
    )

    required_cols = ["path", "minx", "miny", "maxx", "maxy", "res_x", "res_y", "width", "height"]

    missing = [c for c in required_cols if c not in raster_idx.columns]
    if missing:
        raise ValueError(f"Raster index is missing required columns: {missing}")

    for c in ["minx", "miny", "maxx", "maxy", "res_x", "res_y", "width", "height"]:
        raster_idx[c] = pd.to_numeric(raster_idx[c], errors="coerce")

    if "file" not in raster_idx.columns:
        raster_idx["file"] = raster_idx["path"].apply(lambda x: Path(x).name)

    if "folder" not in raster_idx.columns:
        raster_idx["folder"] = raster_idx["path"].apply(lambda x: Path(x).parent.name)

    if "count" not in raster_idx.columns:
        raster_idx["count"] = np.nan

    return raster_idx


FORCE_REBUILD_RASTER_INDEX = False

if RASTER_INDEX_PATH.exists() and not FORCE_REBUILD_RASTER_INDEX:
    raster_idx = pd.read_csv(RASTER_INDEX_PATH)
    print("Loaded existing raster index:", RASTER_INDEX_PATH)
else:
    raster_idx = build_raster_index(DIGIMAP_DIR, RASTER_INDEX_PATH)

raster_idx = standardise_raster_index_columns(raster_idx)

# Save the standardised version back to disk, so later cells use the clean format.
raster_idx.to_csv(RASTER_INDEX_PATH, index=False)

print("raster_idx:", raster_idx.shape)
display(raster_idx.head())

print("Overall bounds:")
print({
    "minx": raster_idx["minx"].min(),
    "miny": raster_idx["miny"].min(),
    "maxx": raster_idx["maxx"].max(),
    "maxy": raster_idx["maxy"].max(),
})

print("Pixel size summary:")
display(raster_idx[["res_x", "res_y", "width", "height"]].describe())

## 5. Prepare crop dataframe

This cell standardises:

- sample ID
- task name: `PTAL` or `EPC`
- BNG centre coordinates
- crop bounds

Default crop sizes:

- PTAL: 300 m × 300 m
- EPC: 150 m × 150 m


In [ ]:
# Merge sample metadata into crop index where possible.
# This keeps the pipeline robust if the crop index only contains coordinates and sample IDs.

df = crop_df.copy()

if "sample_id" not in df.columns:
    df.insert(0, "sample_id", np.arange(len(df)).astype(str))
else:
    df["sample_id"] = df["sample_id"].astype(str)

if "sample_id" in sample_df.columns:
    sample_tmp = sample_df.copy()
    sample_tmp["sample_id"] = sample_tmp["sample_id"].astype(str)

    missing_cols = [c for c in sample_tmp.columns if c not in df.columns]
    if missing_cols:
        df = df.merge(sample_tmp[["sample_id"] + missing_cols], on="sample_id", how="left")
        print("Merged missing columns from sample_df:", len(missing_cols))
elif len(sample_df) == len(df):
    # Fallback: same row order.
    for c in sample_df.columns:
        if c not in df.columns:
            df[c] = sample_df[c].values
    print("Added missing columns from sample_df by row order.")

print("Prepared df:", df.shape)
display(df.head())


In [ ]:
def first_existing_col(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def infer_task_series(df):
    task_col = first_existing_col(df.columns, [
        "task", "dataset", "target", "target_name", "label_type", "sample_type", "source"
    ])

    if task_col is not None:
        s = df[task_col].astype(str).str.upper()
        if s.str.contains("PTAL|EPC", regex=True).any():
            return s.str.extract("(PTAL|EPC)", expand=False)

    for c in ["sample_id", "id", "uid"]:
        if c in df.columns:
            s = df[c].astype(str).str.upper()
            if s.str.contains("PTAL|EPC", regex=True).any():
                return s.str.extract("(PTAL|EPC)", expand=False)

    raise ValueError(
        "Could not infer task type. Please make sure there is a column containing PTAL/EPC."
    )

x_col = first_existing_col(df.columns, [
    "x", "easting", "Easting", "bng_x", "x_bng", "centroid_x", "centre_x", "center_x", "X"
])
y_col = first_existing_col(df.columns, [
    "y", "northing", "Northing", "bng_y", "y_bng", "centroid_y", "centre_y", "center_y", "Y"
])

if x_col is None or y_col is None:
    raise ValueError("Could not find BNG x/y coordinate columns in crop/sample data.")

df["task"] = infer_task_series(df)

if df["task"].isna().any():
    bad = df[df["task"].isna()].head()
    raise ValueError(f"Some rows have unknown task type. Example rows:\n{bad}")

df["x"] = pd.to_numeric(df[x_col], errors="coerce")
df["y"] = pd.to_numeric(df[y_col], errors="coerce")

if df[["x", "y"]].isna().any().any():
    raise ValueError("Some x/y coordinates are missing or non-numeric.")

if df["x"].abs().median() < 1000 or df["y"].abs().median() < 1000:
    raise ValueError(
        "The coordinate columns look like lon/lat, not British National Grid metres. "
        "Please use BNG easting/northing columns."
    )

# Use existing crop bounds if available; otherwise compute from crop size.
left_col = first_existing_col(df.columns, ["crop_left", "left", "xmin", "minx"])
right_col = first_existing_col(df.columns, ["crop_right", "right", "xmax", "maxx"])
bottom_col = first_existing_col(df.columns, ["crop_bottom", "bottom", "ymin", "miny"])
top_col = first_existing_col(df.columns, ["crop_top", "top", "ymax", "maxy"])

has_bounds = all(c is not None for c in [left_col, right_col, bottom_col, top_col])

if has_bounds:
    df["crop_left"] = pd.to_numeric(df[left_col], errors="coerce")
    df["crop_right"] = pd.to_numeric(df[right_col], errors="coerce")
    df["crop_bottom"] = pd.to_numeric(df[bottom_col], errors="coerce")
    df["crop_top"] = pd.to_numeric(df[top_col], errors="coerce")
    print("Using existing crop bounds.")
else:
    crop_size_col = first_existing_col(df.columns, ["crop_size_m", "crop_size", "size_m"])
    if crop_size_col is not None:
        df["crop_size_m"] = pd.to_numeric(df[crop_size_col], errors="coerce")
    else:
        df["crop_size_m"] = np.where(df["task"].eq("PTAL"), 300.0, 150.0)

    half = df["crop_size_m"] / 2.0
    df["crop_left"] = df["x"] - half
    df["crop_right"] = df["x"] + half
    df["crop_bottom"] = df["y"] - half
    df["crop_top"] = df["y"] + half
    print("Computed crop bounds from centre coordinates and crop size.")

if df[["crop_left", "crop_right", "crop_bottom", "crop_top"]].isna().any().any():
    raise ValueError("Some crop bounds are missing or non-numeric.")

print("Task counts:")
display(df["task"].value_counts())

print("Crop size check:")
df["crop_width_m"] = df["crop_right"] - df["crop_left"]
df["crop_height_m"] = df["crop_top"] - df["crop_bottom"]
display(df.groupby("task")[["crop_width_m", "crop_height_m"]].describe())

print("Final crop dataframe:", df.shape)
display(df.head())


## 6. Build crop source index: single-tile or mosaic


In [ ]:
def build_crop_source_index(crop_df, raster_idx, out_path):
    ras = raster_idx.copy()

    for c in ["minx", "miny", "maxx", "maxy"]:
        ras[c] = pd.to_numeric(ras[c], errors="coerce")

    paths = ras["path"].astype(str).to_numpy()
    minx = ras["minx"].to_numpy()
    miny = ras["miny"].to_numpy()
    maxx = ras["maxx"].to_numpy()
    maxy = ras["maxy"].to_numpy()

    rows = []
    eps = 1e-7

    for i, row in tqdm(crop_df.iterrows(), total=len(crop_df), desc="Matching crops to tiles"):
        left = float(row["crop_left"])
        right = float(row["crop_right"])
        bottom = float(row["crop_bottom"])
        top = float(row["crop_top"])

        inside = (
            (minx <= left + eps) &
            (maxx >= right - eps) &
            (miny <= bottom + eps) &
            (maxy >= top - eps)
        )

        inside_idx = np.where(inside)[0]

        if len(inside_idx) > 0:
            tile_idx = int(inside_idx[0])
            tile_list = [paths[tile_idx]]
            source_type = "single_tile"
            tile_path = paths[tile_idx]
        else:
            overlap = (
                (minx < right - eps) &
                (maxx > left + eps) &
                (miny < top - eps) &
                (maxy > bottom + eps)
            )
            overlap_idx = np.where(overlap)[0]

            if len(overlap_idx) == 0:
                tile_list = []
                source_type = "missing"
                tile_path = ""
            else:
                tile_list = [paths[j] for j in overlap_idx]
                source_type = "mosaic"
                tile_path = ""

        base = row.to_dict()
        base.update({
            "source_type": source_type,
            "tile_path": tile_path,
            "tile_paths": json.dumps(tile_list),
            "n_tiles": len(tile_list),
        })
        rows.append(base)

    out = pd.DataFrame(rows)
    out.to_csv(out_path, index=False)
    return out

FORCE_REBUILD_SOURCE_INDEX = False

if SOURCE_INDEX_PATH.exists() and not FORCE_REBUILD_SOURCE_INDEX:
    source_idx = pd.read_csv(SOURCE_INDEX_PATH)
    print("Loaded existing source index:", SOURCE_INDEX_PATH)
else:
    source_idx = build_crop_source_index(df, raster_idx, SOURCE_INDEX_PATH)

print("source_idx:", source_idx.shape)
print("Saved to:", SOURCE_INDEX_PATH)

print("Source type counts:")
display(source_idx["source_type"].value_counts())

print("Task counts:")
display(source_idx["task"].value_counts())

print("Mosaic tile-count breakdown:")
display(source_idx.loc[source_idx["source_type"].eq("mosaic"), "n_tiles"].value_counts().sort_index())

missing = source_idx[source_idx["source_type"].eq("missing")]
print("Missing samples:", len(missing))
if len(missing) > 0:
    display(missing.head())
    raise RuntimeError("Some crops have no intersecting Digimap tile. Stop here and inspect missing rows.")


## 7. Load DINOv2 model

This uses public DINOv2 base:

- model: `facebook/dinov2-base`
- embedding: CLS token
- output dimension: 768


In [ ]:
MODEL_ID = "facebook/dinov2-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)
model = model.to(device)
model.eval()

print("Loaded:", MODEL_ID)


## 8. Define crop and embedding functions


In [ ]:
def parse_tile_paths(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    value = str(value)
    try:
        return json.loads(value)
    except Exception:
        # Fallback for older separator-based strings.
        if "||" in value:
            return value.split("||")
        if value:
            return [value]
        return []

def raster_array_to_pil(arr):
    # arr: bands, height, width
    if arr.ndim != 3:
        raise ValueError(f"Unexpected raster array shape: {arr.shape}")

    if arr.shape[0] >= 3:
        arr = arr[:3, :, :]
    elif arr.shape[0] == 1:
        arr = np.repeat(arr, 3, axis=0)
    else:
        raise ValueError(f"Unsupported number of raster bands: {arr.shape[0]}")

    arr = np.transpose(arr, (1, 2, 0))

    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)

    return Image.fromarray(arr, mode="RGB")

def crop_single_tile(row):
    left = float(row["crop_left"])
    bottom = float(row["crop_bottom"])
    right = float(row["crop_right"])
    top = float(row["crop_top"])

    tile_path = str(row.get("tile_path", ""))
    if tile_path == "" or tile_path.lower() == "nan":
        paths = parse_tile_paths(row["tile_paths"])
        if len(paths) == 0:
            raise ValueError("No tile path available for single_tile row.")
        tile_path = paths[0]

    with rasterio.open(tile_path) as src:
        window = from_bounds(left, bottom, right, top, transform=src.transform)
        arr = src.read(window=window, boundless=True, fill_value=0)

    return raster_array_to_pil(arr)

def crop_mosaic(row):
    left = float(row["crop_left"])
    bottom = float(row["crop_bottom"])
    right = float(row["crop_right"])
    top = float(row["crop_top"])

    paths = parse_tile_paths(row["tile_paths"])
    if len(paths) == 0:
        raise ValueError("No tile paths available for mosaic row.")

    with ExitStack() as stack:
        srcs = [stack.enter_context(rasterio.open(p)) for p in paths]
        arr, _ = merge(
            srcs,
            bounds=(left, bottom, right, top),
            res=(0.25, 0.25),
            indexes=[1, 2, 3],
            nodata=0,
        )

    return raster_array_to_pil(arr)

def load_crop_image(row):
    if row["source_type"] == "single_tile":
        return crop_single_tile(row)
    elif row["source_type"] == "mosaic":
        return crop_mosaic(row)
    else:
        raise ValueError(f"Unsupported source_type: {row['source_type']}")

@torch.inference_mode()
def embed_pil_images(images):
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

    outputs = model(**inputs)

    # DINOv2 CLS token: shape = batch × 768
    emb = outputs.last_hidden_state[:, 0, :]

    emb = emb.detach().float().cpu().numpy()
    return emb

def embed_rows(rows_df, batch_size=16):
    all_embs = []

    for start in tqdm(range(0, len(rows_df), batch_size), desc="Embedding batches", leave=False):
        batch_df = rows_df.iloc[start:start + batch_size]
        images = []

        for _, row in batch_df.iterrows():
            try:
                images.append(load_crop_image(row))
            except Exception as e:
                sid = row.get("sample_id", "unknown")
                raise RuntimeError(f"Failed to load crop for sample_id={sid}") from e

        try:
            emb = embed_pil_images(images)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                raise RuntimeError(
                    "CUDA out of memory. Reduce BATCH_SIZE from 16 to 8 and rerun the failed cell."
                ) from e
            raise e

        all_embs.append(emb)

        del images, emb
        torch.cuda.empty_cache()

    return np.vstack(all_embs)

print("Functions ready.")


## 9. Full DINOv2 embedding extraction

The extraction is resumable:

- chunk size: 500 samples
- default batch size: 16
- if CUDA memory error occurs, set `BATCH_SIZE = 8`
- existing chunk files are skipped automatically


Some chunks have already been generated; the code below is intended to check whether the previously generated results are still there.

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/GEOG0105")
CHUNK_DIR = BASE_DIR / "Outputs" / "embeddings" / "dinov2_aerial_chunks"

chunk_files = sorted(CHUNK_DIR.glob("dinov2_chunk_*.parquet"))

total_rows = 0
for p in chunk_files:
    tmp = pd.read_parquet(p, columns=["sample_id"])
    total_rows += len(tmp)

print("Chunks found:", len(chunk_files))
print("Completed rows:", total_rows)
print("Expected rows:", 26597)
print("Remaining rows:", 26597 - total_rows)
print("Last chunks:", [p.name for p in chunk_files[-5:]])

In [ ]:
CHUNK_SIZE = 500
BATCH_SIZE = 16

EMB_PREFIX = "dinov2_"

# Keep useful metadata in the output.
drop_cols = {"tile_paths"}  # large string column; source_type and n_tiles are kept
meta_cols = [c for c in source_idx.columns if c not in drop_cols]

print("Total samples:", len(source_idx))
print("Chunk size:", CHUNK_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of chunks:", math.ceil(len(source_idx) / CHUNK_SIZE))


In [ ]:
def save_embedding_chunk(chunk_df, out_file, batch_size=16):
    emb = embed_rows(chunk_df, batch_size=batch_size)

    emb_cols = [f"{EMB_PREFIX}{i:04d}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)

    out_df = chunk_df[meta_cols].reset_index(drop=True).copy()
    out_df = pd.concat([out_df, emb_df], axis=1)

    out_df.to_parquet(out_file, index=False)

    return out_df.shape

for chunk_id, start in enumerate(range(0, len(source_idx), CHUNK_SIZE)):
    end = min(start + CHUNK_SIZE, len(source_idx))
    out_file = CHUNK_DIR / f"dinov2_chunk_{chunk_id:04d}.parquet"

    if out_file.exists():
        print(f"Skip existing chunk {chunk_id:04d}: {out_file.name}")
        continue

    chunk_df = source_idx.iloc[start:end].reset_index(drop=True)

    print(f"Processing chunk {chunk_id:04d}: rows {start}–{end - 1}, n={len(chunk_df)}")
    shape = save_embedding_chunk(chunk_df, out_file, batch_size=BATCH_SIZE)
    print(f"Saved {out_file.name}, shape={shape}")

    gc.collect()
    torch.cuda.empty_cache()

print("Full extraction loop finished.")


## 10. Combine DINOv2 chunks

In [ ]:
chunk_files = sorted(CHUNK_DIR.glob("dinov2_chunk_*.parquet"))

chunk_dfs = [
    pd.read_parquet(p)
    for p in chunk_files
]

dinov2_full = pd.concat(chunk_dfs, ignore_index=True)

FINAL_DINO_PATH = EMB_DIR / "dinov2_aerial_embeddings_dino_sample.parquet"

dinov2_full.to_parquet(FINAL_DINO_PATH, index=False)

dinov2_cols = [
    c for c in dinov2_full.columns
    if str(c).startswith("dinov2_")
]

display(dinov2_full.head())

In [ ]:
# DINOv2 summary

summary = {
    "embedding": "DINOv2 aerial embeddings",
    "model": MODEL_ID,
    "output_file": str(FINAL_DINO_PATH),
    "n_rows": int(len(dinov2_full)),
    "n_embedding_features": int(len(dinov2_cols)),
    "task_counts": dinov2_full["task"].value_counts().to_dict(),
    "source_type_counts": dinov2_full["source_type"].value_counts().to_dict(),
    "missing_embedding_values": int(dinov2_full[dinov2_cols].isna().sum().sum())
}

summary_path = TABLE_DIR / "dinov2_aerial_embeddings_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

summary